# Treg-B Interaction

Publication source copy for the Doublets 2 and 4 sequencing analyses. Outputs and execution counts are cleared. See `../VALIDATION_REPORT.md` for the separate execution results and remaining limitations.

Run from the `analysis/` directory after supplying the bundled inputs. Final figure sections explicitly load preserved manuscript checkpoints; they are distinct from fresh clustering.

In [ ]:
from pathlib import Path
Path("../results/Doublets_2_Treg-B").mkdir(parents=True, exist_ok=True)
from threadpoolctl import threadpool_limits
threadpool_limits(limits=12, user_api="blas")


In [ ]:
import pandas as pd
import numpy as np
import scanpy as sc
from anndata import read_h5ad
from anndata import read_text
from collections import Counter
import matplotlib.pyplot as plt
from matplotlib.pyplot import figure
from matplotlib.pyplot import rc_context

import anndata
import seaborn as sns
import regex as re
import os

In [ ]:
sc.settings.verbosity = 3
sc.settings.set_figure_params(dpi=200, dpi_save=300)
sc.settings.n_jobs = 30
pd.options.display.max_columns = None
pd.options.display.max_rows = None

sc.logging.print_header()

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

# Import h5ad with processed data and BCR info

In [ ]:
path_data = '../data/Doublets/data/smartseq_doublets_2_analysis/'

#Ensembl ID with BCR

adata = sc.read_h5ad(path_data+'Doublets_3363cells_GeneSymbols_preprocessed_2xDoubletCounts_wDoubletScores.h5ad')

#the line below fixes an error that results when trying to run sc.tl.rank_genes_groups(adata,'leiden',method='wilcoxon')
#error is: KeyError: 'base'
adata.uns['log1p']["base"] = None
adata

In [ ]:
# This function allows users to access genes using their gene name
# Import all gene names from adata

gene_list = adata.var_names
def find_genes(*genes, mode='exact', printmode=True):
    list = []
    
    # if user passed in a list of genes instead of individual genes
    if type(genes[0]) is not str:
        genes = genes[0]
        
    if mode == 'endswith':
        for g in genes:
            list.extend([i for i in gene_list if i.split()[0].endswith('' + g)])
    if mode == 'exact':
        for g in genes:
            list.extend([i for i in gene_list if i.split('_')[1] == g])
    if mode == 'contains':
        for g in genes:
            list.extend([i for i in gene_list if g in i.split()[0]])    
    if printmode:
        print(list)
    return(list)

# Subset to Tregs

In [ ]:
#subset
list_subset = ['Treg-B-Real','Treg-B-Artificial','Treg','B']

adata = adata[adata.obs['sortedpop'].isin(list_subset)]
adata = adata[adata.obs['SampleID']=='Spleen17']

#filter by doublet score > 0
list_not_doubletscored = ['Treg-B-Artificial','Treg','B']
adata = adata[(adata.obs['sortedpop'].isin(list_not_doubletscored)) | (adata.obs['doubletscore']>0)]

#keep only doublets

adata

# PCA

In [ ]:
#compute pca
sc.tl.pca(adata, svd_solver='arpack')

In [ ]:
sc.pl.pca_scatter(adata, color=['sortedpop'])#, save='_all_ages_droplet_facs_tissue.pdf')
sc.pl.pca_scatter(adata, color=['SampleID'])#, save='_all_ages_droplet_facs_tissue.pdf')
sc.pl.pca(adata, color=['384plate', 'sortedtype', 'sortedpop'], wspace=0.5)
sc.pl.pca_scatter(adata, color=['n_counts', 'n_genes','doubletscore'])

In [ ]:
sc.pl.pca_variance_ratio(adata, log=True)
sc.pl.pca_loadings(adata, include_lowest=True, components = '1,2,3,4,5,6')

# Cluster

In [ ]:
# Compute neighborhood graph followed by UMAP
sc.pp.neighbors(adata)
sc.tl.umap(adata)

#recommended method: Leiden graph clustering

sc.tl.leiden(adata,resolution=0.4)
#

In [ ]:
# Plot metadata
sc.pl.umap(adata, color=['sortedtype','sortedpop'], wspace = 0.4)
sc.pl.umap(adata, color=['leiden','doubletscore'], legend_loc='on data')

#
df_grouped = adata.obs.groupby('leiden')['sortedpop'].value_counts(normalize=True).unstack()
df_grouped.plot.bar(stacked=True)
plt.legend(
    bbox_to_anchor=(0.5, 1.02),
    loc="lower center",
    borderaxespad=0,
    frameon=False,
    ncol=3,
)
plt.grid(False)
plt.show()

df_grouped = adata.obs.groupby('sortedpop')['leiden'].value_counts(normalize=True).unstack()
df_grouped.plot.bar(stacked=True)
plt.legend(
    bbox_to_anchor=(0.5, 1.02),
    loc="lower center",
    borderaxespad=0,
    frameon=False,
    ncol=3,
)
plt.grid(False)
plt.show()

# Rename leiden clusters

In [ ]:
new_cluster_names = ['No Intx',
                     'B',
                     'Treg-B Intx GC',
                     'Treg',
                     'Treg-B Intx Cytotoxic']
adata.rename_categories('leiden', new_cluster_names)

# adata

In [ ]:
sc.tl.rank_genes_groups(adata, groupby='leiden', method='logreg', tie_correct=True, pts=True, key_added='logreg')
cluster_genes = pd.DataFrame(adata.uns['logreg']['names'])
cluster_genes.head(100)

In [ ]:
adata

# Save processed Treg-B h5ad

In [ ]:
# #save
adata.write_h5ad(path_data+'Doublets_SP17_629cells_TregB_processed_2xDoubletCounts.h5ad')

# Load saved file

In [ ]:
adata = sc.read_h5ad(path_data+'Doublets_SP17_629cells_TregB_processed_2xDoubletCounts.h5ad')
adata

# Manuscript figures 202410

In [ ]:
# Use the preserved manuscript object for the published figure layout.
path_data = '../data/Doublets/data/smartseq_doublets_2_analysis/'
adata = sc.read_h5ad('../data/reference_checkpoints/Doublets_SP17_629cells_TregB_processed_2xDoubletCounts.h5ad')

adata.uns['log1p']["base"] = None
adata

In [ ]:
# #rename leiden clusters
new_cluster_names = ['No Intx',
                     'B',
                     'Th1/Tfh - MZB Intx',
                     'Treg/Tfr',
                     'Cytotoxic Treg - B Intx',
                    ]
adata.rename_categories('leiden', new_cluster_names)

#rename sortedpop categories

dict_sortedpop_old_to_new = {'Treg-B-Real': 'CD25+ T-B',
         'Treg-B-Artificial': 'Artificial Treg-B', 
         'Treg': 'Treg',
         'B': 'B'}

adata.obs['sortedpop'] = (
    adata.obs['sortedpop']
    .map(dict_sortedpop_old_to_new)
    .astype('category')
)

#edit names
dict_leiden_old_to_new = {'No Intx':'T-B non-interacting',
                            'Cytotoxic Treg - B Intx':'Cytotoxic Treg-B',
                            'Th1/Tfh - MZB Intx':'Th1/Tfh-MZB',
                            'B':'B',
                            'Treg/Tfr':'Treg/Tfr'}
adata.obs['leiden'] = (
    adata.obs['leiden']
    .map(dict_leiden_old_to_new)
    .astype('category')
)

In [ ]:
Counter(adata.obs['leiden'])

In [ ]:
path_figures = '../results/Doublets_2_Treg-B/'
sc.settings.figdir = path_figures
sc.set_figure_params(dpi=300, dpi_save=600)

leiden_order_1 = ['Th1/Tfh-MZB',
                  'Cytotoxic Treg-B',
                  'T-B non-interacting',
                  'B',
                  'Treg/Tfr']

leiden_order_2 = leiden_order_1[::-1]

sortedpop_order_1 = ['CD25+ T-B',
                     'Artificial Treg-B',
                     'Treg',
                     'B']
sortedpop_order_2 = sortedpop_order_1[::-1]

In [ ]:
#color palette
leiden_palette_2 = [
    
    "#FED976",  # Yellow
    
    "#FD8D3C",  # Orange

    "#969696",  # Medium Gray
    
    "#B2182B",  # Dark Red
    
    "#3182BD",  # Blue

]

In [ ]:
#all 
adata_plot_leiden = adata.copy()
adata_plot_leiden.obs['leiden'] = adata_plot_leiden.obs['leiden'].cat.set_categories(leiden_order_2, ordered=True)
sc.pl.umap(adata_plot_leiden, color='leiden', legend_fontsize=12, frameon=False, title='', palette=leiden_palette_2, save='_Fig_S3_SP17_leiden.pdf')

#singlets
adata_plot_leiden_singlet = adata.copy()
sortedpop_order_2_singlets = sortedpop_order_2[0:2]
adata_plot_leiden_singlet.obs['leiden'] = [x if y in sortedpop_order_2_singlets else 'none' for x,y in zip(adata_plot_leiden_singlet.obs['leiden'],adata_plot_leiden_singlet.obs['sortedpop'])] #mask to color leiden clusters by sorted population
adata_plot_leiden_singlet.obs['leiden'] = adata_plot_leiden_singlet.obs['leiden'].astype('category') #stop error
adata_plot_leiden_singlet.obs['leiden'] = adata_plot_leiden_singlet.obs['leiden'].cat.set_categories(leiden_order_2, ordered=True)
sc.pl.umap(adata_plot_leiden_singlet, color='leiden', legend_fontsize=12, frameon=False, title='Singlets', palette=leiden_palette_2, na_color='whitesmoke', legend_loc = None, save='_Fig_S3_SP17_leiden_Singlets.pdf')

#artificial doublets
adata_plot_leiden_artdoub = adata.copy()
sortedpop_order_2_artdoub = sortedpop_order_2[2:3]
adata_plot_leiden_artdoub.obs['leiden'] = [x if y in sortedpop_order_2_artdoub else 'none' for x,y in zip(adata_plot_leiden_artdoub.obs['leiden'],adata_plot_leiden_artdoub.obs['sortedpop'])] #mask to color leiden clusters by sorted population
adata_plot_leiden_artdoub.obs['leiden'] = adata_plot_leiden_artdoub.obs['leiden'].astype('category') #stop error
adata_plot_leiden_artdoub.obs['leiden'] = adata_plot_leiden_artdoub.obs['leiden'].cat.set_categories(leiden_order_2, ordered=True)
sc.pl.umap(adata_plot_leiden_artdoub, color='leiden', legend_fontsize=12, frameon=False, title='Non-interacting Controls', palette=leiden_palette_2, na_color='whitesmoke', legend_loc = None, save='_Fig_S3_SP17_leiden_ArtDoub.pdf')

#real doublets
adata_plot_leiden_realdoub = adata.copy()
sortedpop_order_2_realdoub = sortedpop_order_2[3:4]
adata_plot_leiden_realdoub.obs['leiden'] = [x if y in sortedpop_order_2_realdoub else 'none' for x,y in zip(adata_plot_leiden_realdoub.obs['leiden'],adata_plot_leiden_realdoub.obs['sortedpop'])] #mask to color leiden clusters by sorted population
adata_plot_leiden_realdoub.obs['leiden'] = adata_plot_leiden_realdoub.obs['leiden'].astype('category') #stop error
adata_plot_leiden_realdoub.obs['leiden'] = adata_plot_leiden_realdoub.obs['leiden'].cat.set_categories(leiden_order_2, ordered=True)
sc.pl.umap(adata_plot_leiden_realdoub, color='leiden', legend_fontsize=12, frameon=False, title='Contacting Pairs', palette=leiden_palette_2, na_color='whitesmoke', legend_loc = None, save='_Fig_S3_SP17_leiden_RealDoub.pdf')

In [ ]:
## Plot distribution of sorted populations
df_grouped = adata.obs.groupby('sortedpop')['leiden'].value_counts(normalize=True).unstack()
df_grouped = df_grouped.reindex(sortedpop_order_2) #order rows
df_grouped = df_grouped[leiden_order_2] #order columns

#singlet

#singlet
df_grouped.loc[sortedpop_order_2[0:2]].plot.barh(stacked=True, color=leiden_palette_2, legend=False)
plt.grid(False)
plt.ylabel('')
sns.despine()
plt.savefig(path_figures+'barplot_Fig_S3_SP17_singlet.pdf', bbox_inches='tight')
plt.show()

# artificial doublet
df_grouped.loc[sortedpop_order_2[2:3]].plot.barh(stacked=True, color=leiden_palette_2, legend=False)
plt.grid(False)
plt.ylabel('')
sns.despine()
plt.savefig(path_figures+'barplot_Fig_S3_SP17_ArtDoub.pdf', bbox_inches='tight')
plt.show()

# real doublet
df_grouped.loc[sortedpop_order_2[3:4]].plot.barh(stacked=True, color=leiden_palette_2, legend=False)
plt.grid(False)
plt.ylabel('')
sns.despine()
plt.savefig(path_figures+'barplot_Fig_S3_SP17_RealDoub.pdf', bbox_inches='tight')

plt.show()

In [ ]:
list_plot_genes = ['CD3E', 'MS4A1', 'FOXP3', 'IL2RA', 'TNFRSF18', 'PRF1', 'GZMA', 'GZMB', 'BCL6', 'AICDA', 'CD1C', 'TBX21', 'IL12A', 'IL12B', 'CXCR3', 'NRN1']

with plt.rc_context():  # Use this to set figure params like size and dpi
    #scaled
    sc.pl.dotplot(adata, list_plot_genes, 'leiden', dendrogram=False, cmap='Reds', standard_scale='var', colorbar_title='column scaled\nexpression',show=True, swap_axes=False, categories_order=leiden_order_2, save='Fig_S3_SP17.pdf')